# Week 8, Day 3: The chain rule: composing functions and layers

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w008/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Derivatives of a model neuron

For $r = \sigma(wx + b)$, write `dr_dx`, `dr_dw` and `dr_db` using the chain rule, and check each against a central difference. Then print all three at x = 0.5, w = 2, b = −1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def sigmoid(u): return 1 / (1 + np.exp(-u))
def dsigmoid(u): return sigmoid(u) * (1 - sigmoid(u))
def neuron(x, w, b): return sigmoid(w * x + b)

def dr_dx(x, w, b):
    return ...
def dr_dw(x, w, b):
    return ...
def dr_db(x, w, b):
    return ...

print(dr_dx(0.5, 2.0, -1.0), dr_dw(0.5, 2.0, -1.0), dr_db(0.5, 2.0, -1.0))

In [ ]:
import numpy as np
def num(f, a, h=1e-6):
    return (f(a + h) - f(a - h)) / (2 * h)

def test_dx():
    "dr_dx matches a numerical derivative"
    for x, w, b in [(0.5, 2.0, -1.0), (-1.0, 0.7, 0.3), (2.0, -1.5, 0.0)]:
        assert np.isclose(dr_dx(x, w, b), num(lambda t: neuron(t, w, b), x))

def test_dw_db():
    "dr_dw and dr_db match numerical derivatives"
    for x, w, b in [(0.5, 2.0, -1.0), (-1.0, 0.7, 0.3)]:
        assert np.isclose(dr_dw(x, w, b), num(lambda t: neuron(x, t, b), w))
        assert np.isclose(dr_db(x, w, b), num(lambda t: neuron(x, w, t), b))

test_dx()
test_dw_db()
print("All checks passed")

### Exercise: Products of many derivatives

Write `chain_derivative(f, df, h0, n)` that applies `f` n times starting from `h0` and returns the product of the local derivatives `df(h)` evaluated at each stage's input. Use it for 1 to 30 stages of a sigmoid, a ReLU, and linear stages with weight 1.2 and 0.8. Plot the magnitudes on a log scale.

In [ ]:
def chain_derivative(f, df, h0, n):
    h, total = h0, 1.0
    for _ in range(n):
        ...
    return total

ns = range(1, 31)
d_sig = [chain_derivative(sigmoid, dsigmoid, 0.5, n) for n in ns]
d_relu = [chain_derivative(lambda h: np.maximum(0, h), lambda h: float(h > 0), 0.5, n) for n in ns]
d_up = [chain_derivative(lambda h: 1.2 * h, lambda h: 1.2, 0.5, n) for n in ns]
d_down = [chain_derivative(lambda h: 0.8 * h, lambda h: 0.8, 0.5, n) for n in ns]

plt.figure(figsize=(5.5, 3.3))
for d, lab in [(d_sig, "sigmoid"), (d_relu, "ReLU"), (d_up, "linear, w = 1.2"), (d_down, "linear, w = 0.8")]:
    plt.semilogy(list(ns), d, label=lab)
plt.xlabel("number of stages"); plt.ylabel("|dh_L / dh_0|"); plt.legend(frameon=False); plt.show()
print("after 20 sigmoid stages:", d_sig[19])

In [ ]:
import numpy as np
def test_fn():
    "chain_derivative returns the product of local derivatives over n stages"
    d = chain_derivative(lambda h: 2 * h, lambda h: 2.0, 0.3, 5)
    assert np.isclose(d, 32)

def test_vanish():
    "through 20 sigmoid stages the derivative is below 1e-12; through 20 ReLU stages (positive start) it is exactly 1"
    assert d_sig[19] < 1e-12
    assert np.isclose(d_relu[19], 1)

def test_linear():
    "linear stages with weight 1.2 explode and with weight 0.8 vanish"
    assert d_up[19] > 30 and d_down[19] < 0.02

test_fn()
test_vanish()
test_linear()
print("All checks passed")

### Exercise: A forward and a backward pass

With x = 1.5, w = 0.4 and target y = 0.9, compute the forward values `u`, `r`, `L`. Then go backwards: `dL_dr` (the derivative of the squared error), `dL_du` (multiply by the sigmoid's slope) and `dL_dw` (multiply by the derivative of u with respect to w). Check against a numerical derivative and take one gradient step.

In [ ]:
x, w, y = 1.5, 0.4, 0.9
u = ...
r = ...
L = ...
dL_dr = ...
dL_du = ...
dL_dw = ...

w_new = w - 1.0 * dL_dw                      # one gradient-descent step with learning rate 1
L_new = 0.5 * (sigmoid(w_new * x) - y) ** 2
print(f"loss {L:.4f} -> {L_new:.4f};  dL/dw = {dL_dw:.4f}")

In [ ]:
import numpy as np
def test_forward():
    "u, r and L hold the forward values"
    assert np.isclose(u, w * x) and np.isclose(r, sigmoid(u)) and np.isclose(L, 0.5 * (r - y) ** 2)

def test_backward():
    "dL_dr, dL_du and dL_dw are the backward products"
    assert np.isclose(dL_dr, r - y)
    assert np.isclose(dL_du, dL_dr * dsigmoid(u))
    assert np.isclose(dL_dw, dL_du * x)

def test_numeric():
    "dL_dw agrees with a central difference"
    Lw = lambda ww: 0.5 * (sigmoid(ww * x) - y) ** 2
    assert np.isclose(dL_dw, (Lw(w + 1e-6) - Lw(w - 1e-6)) / 2e-6)

test_forward()
test_backward()
test_numeric()
print("All checks passed")